# Hlawga linked-fauna cleanup

This audit aligns Hlawga Wildlife Park's visible `တွေ့ရှိရသော သတ္တဝါများ` data with animal records whose stored protected-area relationship explicitly names Hlawga.


## Result

- Explicitly linked animals: `မျောက်စပ်` (Rhesus Macaque, ID 59) and `တောကြောင်` (Jungle Cat, ID 37).
- `species_fauna` was reduced from five identifiers to the two linked identifiers.
- `species_fauna_mm` now contains only `မျောက်စပ်,တောကြောင်`.
- `other_fauna` and `other_fauna_mm` were cleared because they contained generic or unlinked entries.


In [ ]:
import json, sqlite3
from pathlib import Path

slug = 'hlawga_wildlife_park_protected'
records = json.loads(Path('../data/protected_areas.json').read_text(encoding='utf-8-sig'))
hlawga = next(record for record in records if record['slug'] == slug)

db = sqlite3.connect('file:../wildlife.db?mode=ro', uri=True)
db.row_factory = sqlite3.Row
linked = []
for animal in db.execute('SELECT id, name_my, name_en, scientific_name, protected_areas_my, portal_data FROM animals ORDER BY id'):
    portal = json.loads(animal['portal_data'] or '{}')
    relationship_text = (animal['protected_areas_my'] or '') + ' ' + str(portal.get('in_protected_area') or '')
    if 'hlawga wildlife park' in relationship_text.lower() or 'လှော်ကား ဥယျာဥ်' in relationship_text:
        linked.append(dict(animal))

[(row['id'], row['name_my'], row['name_en'], row['scientific_name']) for row in linked], {
    field: hlawga.get(field)
    for field in ('species_fauna', 'species_fauna_mm', 'other_fauna', 'other_fauna_mm')
}
